<img src="https://i.postimg.cc/J7sf3Lq4/Group.png" alt="FUTURA" width="220" />

# Taller 1 — Web Scraping con **Selenium**

En este taller vamos a:

1. **Configurar** un navegador automatizado (Chrome) controlado por Python.
2. **Inspeccionar** una página web (mi CV personal) y extraer datos específicos.
3. **Recorrer** múltiples páginas (Quotes to Scrape) navegando link por link.
4. **Almacenar** los resultados en un `DataFrame` de pandas para análisis.

> 💡 **Concepto clave:** Selenium **maneja un navegador real** (a diferencia de `requests`, que solo descarga HTML). Eso lo hace más lento, pero permite scrapear sitios con JavaScript, formularios, login, scroll infinito, etc.

## 0 · Instalación de librerías

Solo se ejecuta **la primera vez**. Después podemos comentarlas.

In [ ]:
## INSTALAR SELENIUM
# !pip install selenium
# !pip install webdriver-manager #Instalar ChromeDriver

## 1 · Importar las librerías necesarias

| Import | Para qué sirve |
|---|---|
| `webdriver` | Es el objeto principal: representa el navegador. |
| `Service` | Le dice a Selenium **dónde está** el ejecutable de ChromeDriver. |
| `ChromeDriverManager` | Descarga e instala ChromeDriver automáticamente. |
| `By` | Define **cómo** vamos a buscar elementos (XPath, CSS, ID, clase, etc.). |

In [1]:
# IMPORTAR LIBRERÍAS
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

## 2 · Configurar el driver (el navegador)

Antes de abrir Chrome, le pasamos **opciones** para personalizar su comportamiento:

| Opción | Qué hace |
|---|---|
| `--start-maximized` | Abre la ventana maximizada (más cómodo para inspeccionar). |
| `--disable-notifications` | Bloquea pop-ups de notificaciones del navegador. |
| `--no-sandbox` | Necesario en algunos entornos (Docker, Linux). |
| `--verbose` | Muestra logs detallados (útil para debuggear). |
| `prefs notifications: 2` | `2 = bloquear`. Refuerza el bloqueo de notificaciones del sitio. |
| `--incognito` *(opcional)* | Modo incógnito: no guarda cookies ni historial. |

> ⚠️ **Importante:** este `driver` queda **abierto** y lo vamos a reutilizar en todo el notebook. Al final del taller, conviene hacer `driver.quit()`.

In [62]:
# CONFIGURACIÓN DEL DRIVER
options = webdriver.ChromeOptions()
# options.add_argument('--incognito') # modo incógnito
options.add_argument('--start-maximized')
options.add_argument('--disable-notifications')
options.add_argument('--no-sandbox')
options.add_argument('--verbose')
prefs = {"profile.default_content_setting_values.notifications" : 2}

options.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()),
                            options = options)

## 3 · Inspeccionando un CV

Vamos a entrar a una página web sencilla (mi CV) y extraer:

- **Nombre completo** (título principal)
- **Link de contacto** (LinkedIn)
- **Universidad** (link)
- **Skills técnicas** (lista)

### 🔍 ¿Cómo encontramos los XPath?
1. Abrimos la página en Chrome.
2. Click derecho sobre el elemento → **Inspeccionar**.
3. En el HTML, click derecho → **Copy → Copy XPath** (o lo escribimos a mano para que sea más robusto).

In [3]:
# Ingresamos a la web
url = 'https://carlosrondanp.github.io/'
driver.get(url)

### 3.1 · Definimos los XPath y extraemos los datos

**Métodos clave de Selenium:**

| Método | Devuelve | Cuándo usarlo |
|---|---|---|
| `find_element(By.XPATH, ...)` | **UN** elemento (el primero que encuentre) | Cuando sabés que hay 1 solo (ej: el título) |
| `find_elements(...)` (con S) | **LISTA** de elementos | Cuando hay varios (ej: skills, ítems) |
| `.text` | El texto visible del elemento | Para `<span>Python</span>` → `"Python"` |
| `.get_attribute('href')` | El valor de un atributo HTML | Para sacar el link de un `<a href="...">` |

In [4]:
# Ubicamos los elementos a extraer
nombre_xpath = "//h1[@id='nombre']"
contacto_xpath = "//p[@id='contacto']/a[1]"
uni_xpath = "//div[@id='perfil']/p/a"

skills_xpath = "//span[@class='badge']"
skills_1_xpath = "//span[@class='badge'][1]"

nombre = driver.find_element(By.XPATH, nombre_xpath).text
contacto = driver.find_element(By.XPATH, contacto_xpath).get_attribute('href')
link_uni = driver.find_element(By.XPATH , uni_xpath).get_attribute('href')
skills = driver.find_elements(By.XPATH, skills_xpath)
skills = [skill.text for skill in skills]

### 3.2 · Verificamos los resultados

In [5]:
driver.find_element(By.XPATH, nombre_xpath).text

'Carlos Enrique Rondan Poma'

In [6]:
nombre

'Carlos Enrique Rondan Poma'

In [7]:
contacto

'mailto:carlos.rondan.poma@gmail.com'

In [8]:
link_uni

'https://portal.uni.edu.pe/'

In [9]:
skills

['Python', 'SQL', 'R', 'C++', 'AWS', 'GCP', 'Power BI', 'Tableau']

## 4 · Quotes to Scrape

Ahora vamos a scrapear un sitio pensado para practicar: <https://quotes.toscrape.com/>

**Objetivo:**
1. Extraer las **citas**, sus **autores** y **tags** de la primera página.
2. Tomar el link de cada autor y **navegar** a su página personal.
3. En cada página de autor, extraer **fecha** y **lugar de nacimiento**.
4. Guardar todo en un `DataFrame`.

In [63]:
# Ingresamos a la web
url = 'https://quotes.toscrape.com/'
driver.get(url)

### 4.1 · Extraemos tags, autores y links de la primera página

**Truco útil — `following-sibling`:**  
El XPath `//small[@class='author']/following-sibling::a` significa:  
*"Buscame el `<small>` con clase 'author', y desde ahí, agarrá el siguiente `<a>` que sea **hermano** suyo."*  

Esto sirve cuando dos elementos están "al lado" en el HTML, no anidados.

In [64]:
primer_tag_xpath = '//div[@class="tags"]//a[1]'
autores_xpath = '//small[@class="author"][1]'
links_autores_xpath = '//small[@class="author"]/following-sibling::a'

In [65]:
tags = [tag.text for tag in driver.find_elements(By.XPATH, primer_tag_xpath)]
autores = [autor.text for autor in driver.find_elements(By.XPATH, autores_xpath)]
links_autores = [link_autor.get_attribute('href') for link_autor in driver.find_elements(By.XPATH, links_autores_xpath)]

In [67]:
tags

['change',
 'abilities',
 'inspirational',
 'aliteracy',
 'be-yourself',
 'adulthood',
 'life',
 'edison',
 'misattributed-eleanor-roosevelt',
 'humor']

In [68]:
autores

['Albert Einstein',
 'J.K. Rowling',
 'Albert Einstein',
 'Jane Austen',
 'Marilyn Monroe',
 'Albert Einstein',
 'André Gide',
 'Thomas A. Edison',
 'Eleanor Roosevelt',
 'Steve Martin']

In [73]:
links_autores

['https://quotes.toscrape.com/author/Albert-Einstein',
 'https://quotes.toscrape.com/author/J-K-Rowling',
 'https://quotes.toscrape.com/author/Albert-Einstein',
 'https://quotes.toscrape.com/author/Jane-Austen',
 'https://quotes.toscrape.com/author/Marilyn-Monroe',
 'https://quotes.toscrape.com/author/Albert-Einstein',
 'https://quotes.toscrape.com/author/Andre-Gide',
 'https://quotes.toscrape.com/author/Thomas-A-Edison',
 'https://quotes.toscrape.com/author/Eleanor-Roosevelt',
 'https://quotes.toscrape.com/author/Steve-Martin']

In [70]:
import time

### 4.2 · Navegamos a cada autor y extraemos su biografía

**Patrón clásico de scraping:**

```
list_data = []                  ← lista vacía donde vamos guardando
for cada_link en links:
    driver.get(cada_link)       ← abrimos el link
    extraer datos del HTML
    guardar en un dict
    list_data.append(dict)      ← agregamos a la lista
```

Al final, esa lista de diccionarios la convertimos directo en `DataFrame` con `pd.DataFrame(list_data)`.

**Truco útil — XPath relativo:**  
El XPath `'./following-sibling::span[1]'` empieza con `.` → significa **"a partir de este elemento"**, no del documento entero. Por eso lo usamos como `born.find_element(By.XPATH, ubicacion_xpath)` (búsqueda **dentro** de `born`).

In [36]:
import pandas as pd

In [91]:
list_data = list()
for link in links_autores:
    dicc = dict()
    driver.get(link)
    
    ## EXTRAER INFORMACION
    nombre_xpath = '//h3'
    date_xpath = '//span[@class="author-born-date"]'
    location_xpath = '//span[@class="author-born-location"]'
    
    # GUARDAR INFORMACIÓN
    nombre = driver.find_elements(By.XPATH, nombre_xpath)[0].text
    date = driver.find_elements(By.XPATH, date_xpath)[0].text
    location = driver.find_elements(By.XPATH, location_xpath)[0].text

    dicc["Nombre"] = nombre
    dicc["Fecha_Nacimiento"] = date
    dicc["Lugar_Nacimiento"] = location
    dicc["Link"] = link

    list_data.append(dicc)

dataframe = pd.DataFrame(list_data)
dataframe

,Nombre,Fecha_Nacimiento,Lugar_Nacimiento,Link
0,Albert Einstein,"March 14, 1879","in Ulm, Germany",https://quotes.toscrape.com/author/Albert-Eins...
1,J.K. Rowling,"July 31, 1965","in Yate, South Gloucestershire, England, The U...",https://quotes.toscrape.com/author/J-K-Rowling
2,Albert Einstein,"March 14, 1879","in Ulm, Germany",https://quotes.toscrape.com/author/Albert-Eins...
3,Jane Austen,"December 16, 1775","in Steventon Rectory, Hampshire, The United Ki...",https://quotes.toscrape.com/author/Jane-Austen
4,Marilyn Monroe,"June 01, 1926",in The United States,https://quotes.toscrape.com/author/Marilyn-Monroe
5,Albert Einstein,"March 14, 1879","in Ulm, Germany",https://quotes.toscrape.com/author/Albert-Eins...
6,André Gide,"November 22, 1869","in Paris, France",https://quotes.toscrape.com/author/Andre-Gide
7,Thomas A. Edison,"February 11, 1847","in Milan, Ohio, The United States",https://quotes.toscrape.com/author/Thomas-A-Ed...
8,Eleanor Roosevelt,"October 11, 1884",in The United States,https://quotes.toscrape.com/author/Eleanor-Roo...
9,Steve Martin,"August 14, 1945","in Waco, Texas, The United States",https://quotes.toscrape.com/author/Steve-Martin


### 4.3 · Verificamos las listas que generamos antes

In [39]:
autores

['Albert Einstein',
 'J.K. Rowling',
 'Albert Einstein',
 'Jane Austen',
 'Marilyn Monroe',
 'Albert Einstein',
 'André Gide',
 'Thomas A. Edison',
 'Eleanor Roosevelt',
 'Steve Martin']

In [88]:
tags

['change',
 'abilities',
 'inspirational',
 'aliteracy',
 'be-yourself',
 'adulthood',
 'life',
 'edison',
 'misattributed-eleanor-roosevelt',
 'humor']

In [41]:
links_autores

['https://quotes.toscrape.com/author/Albert-Einstein',
 'https://quotes.toscrape.com/author/J-K-Rowling',
 'https://quotes.toscrape.com/author/Albert-Einstein',
 'https://quotes.toscrape.com/author/Jane-Austen',
 'https://quotes.toscrape.com/author/Marilyn-Monroe',
 'https://quotes.toscrape.com/author/Albert-Einstein',
 'https://quotes.toscrape.com/author/Andre-Gide',
 'https://quotes.toscrape.com/author/Thomas-A-Edison',
 'https://quotes.toscrape.com/author/Eleanor-Roosevelt',
 'https://quotes.toscrape.com/author/Steve-Martin']

## 5 · Cerrar el navegador (importante)

Cuando terminamos, conviene cerrar Chrome para liberar recursos. Si no lo hacemos, queda un proceso abierto en segundo plano.

In [92]:
# Cierra TODAS las ventanas del navegador y termina el proceso de ChromeDriver
driver.quit()

## 📌 Resumen de lo aprendido

| Concepto | Método / Sintaxis |
|---|---|
| Abrir el navegador | `webdriver.Chrome(service=..., options=...)` |
| Ir a una URL | `driver.get(url)` |
| Buscar UN elemento | `driver.find_element(By.XPATH, xpath)` |
| Buscar VARIOS elementos | `driver.find_elements(By.XPATH, xpath)` |
| Sacar el texto visible | `.text` |
| Sacar un atributo (link, etc.) | `.get_attribute('href')` |
| Búsqueda relativa (dentro de otro elemento) | XPath que empieza con `.` |
| Cerrar el navegador | `driver.quit()` |